# Data quality and profiling

Generate the full ledger with `python -m scripts.pipeline` from the repository root. These outputs reconcile the analytical snapshot to that database and its offline truth ledger.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

root = Path.cwd()
if not (root / "deployment").exists():
    root = root.parent
snapshot = root / "deployment/snapshot"
profile = json.loads((snapshot / "data_report.json").read_text())
model = json.loads((snapshot / "model.json").read_text())
def sql(name):
    return pd.read_csv(snapshot / "analytics" / f"{name}.csv")


In [2]:
display(pd.Series(profile["summary"], name="value").to_frame())
display(pd.Series(profile["checks"], name="violations").to_frame())
assert not any(profile["checks"].values())
display(sql("reconciliation"))
assert sql("reconciliation").difference.eq(0).all()


,value
claims,120000
states,8
vendors,60
closure_rate,1.0
settlement_days,31.972892
first_report,2021-01-01
last_report,2024-12-31


,violations
chronology,0
event timing,0
payments,0
reserve continuity,0
reserve closure,0
reversals,0
intervention chronology,0
payment adjustment references,0
duplicate reversals,0
financial reconciliation,0


,legitimate,observed_paid,avoidable,missed_recovery,difference
0,3.120275e+08,3.162497e+08,4661875.07,439672.66,0.0


In [3]:
display(pd.Series(profile["descriptive"]["feature_missingness"],name="missing").to_frame())
assert profile["descriptive"]["duplicate_claim_ids"] == 0
display(pd.Series(profile["descriptive"]["paid_quantiles"],name="paid dollars").to_frame())


,missing
claim_id,0
checkpoint,0
coverage,0
severity,0
complexity,0
estimate,0
reserve,0
report_delay,0
workload,0
documented,0


,paid dollars
0.1,199.1350
0.5,1760.8850
0.9,6022.0600
0.99,13606.7983


DuckDB enforces primary and foreign keys on insertion. The reconciliation uses posted amounts net of reversals and separates missed recovery from observed paid. The simulated cohort is retrospectively mature, so its closure rate describes the extract design.